In [1]:
!pip install lightgbm catboost xgboost -q

import os, glob, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
!ls /kaggle/input/datasets/tymofiivoitekh/

data-mining-2026-asg-3


In [ ]:
TRAIN_DIR = Path(
    '/kaggle/input/datasets/tymofiivoitekh/data-mining-2026-asg-3/train/train')
TEST_DIR = Path(
    '/kaggle/input/datasets/tymofiivoitekh/data-mining-2026-asg-3/test/test')
SAMPLE = Path(
    '/kaggle/input/datasets/tymofiivoitekh/data-mining-2026-asg-3/sample_submission.csv')

# verify
print(len(list(TRAIN_DIR.rglob('*.csv'))), 'train files')
print(len(list(TEST_DIR.rglob('*.csv'))),  'test files')

11020 train files
6849 test files


In [ ]:
from scipy import stats


def extract_features(df):
    feats = {}
    axes = ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']

    # ── Basic stats on all 6 columns ──────────────────────────
    for col in axes:
        s = df[col].values
        feats[f'{col}_mean'] = np.mean(s)
        feats[f'{col}_std'] = np.std(s)
        feats[f'{col}_min'] = np.min(s)
        feats[f'{col}_max'] = np.max(s)
        feats[f'{col}_range'] = np.ptp(s)
        feats[f'{col}_median'] = np.median(s)
        feats[f'{col}_q25'] = np.percentile(s, 25)
        feats[f'{col}_q75'] = np.percentile(s, 75)
        feats[f'{col}_iqr'] = np.percentile(s, 75) - np.percentile(s, 25)
        feats[f'{col}_skew'] = stats.skew(s)
        feats[f'{col}_kurt'] = stats.kurtosis(s)
        feats[f'{col}_energy'] = np.mean(s**2)
        feats[f'{col}_rms'] = np.sqrt(np.mean(s**2))
        feats[f'{col}_mad'] = np.mean(np.abs(s - np.mean(s)))
        feats[f'{col}_zero_cr'] = np.mean(np.diff(np.sign(s)) != 0)
        feats[f'{col}_entropy'] = stats.entropy(np.abs(s) + 1e-9)
        feats[f'{col}_slope'] = np.polyfit(np.arange(len(s)), s, 1)[0]
        if np.std(s) > 0 and len(s) > 1:
            feats[f'{col}_autocorr1'] = np.corrcoef(s[:-1], s[1:])[0, 1]
        else:
            feats[f'{col}_autocorr1'] = 0.0

    # ── Magnitude ─────────────────────────────────────────────
    mx = df['mean_x'].values
    my = df['mean_y'].values
    mz = df['mean_z'].values
    mag = np.sqrt(mx**2 + my**2 + mz**2)

    for name, arr in [('mag', mag)]:
        feats[f'{name}_mean'] = np.mean(arr)
        feats[f'{name}_std'] = np.std(arr)
        feats[f'{name}_max'] = np.max(arr)
        feats[f'{name}_min'] = np.min(arr)
        feats[f'{name}_range'] = np.ptp(arr)
        feats[f'{name}_energy'] = np.mean(arr**2)
        feats[f'{name}_skew'] = stats.skew(arr)
        feats[f'{name}_kurt'] = stats.kurtosis(arr)
        feats[f'{name}_entropy'] = stats.entropy(arr + 1e-9)
        feats[f'{name}_iqr'] = np.percentile(arr, 75) - np.percentile(arr, 25)
        feats[f'{name}_q25'] = np.percentile(arr, 25)
        feats[f'{name}_q75'] = np.percentile(arr, 75)

    # ── Jerk (derivative of acceleration) ─────────────────────
    for col, arr in [('x', mx), ('y', my), ('z', mz), ('mag', mag)]:
        jerk = np.diff(arr)
        feats[f'jerk_{col}_mean'] = np.mean(np.abs(jerk))
        feats[f'jerk_{col}_std'] = np.std(jerk)
        feats[f'jerk_{col}_max'] = np.max(np.abs(jerk))
        feats[f'jerk_{col}_energy'] = np.mean(jerk**2)
        feats[f'jerk_{col}_kurt'] = stats.kurtosis(jerk)

    # ── Cross-axis correlations ────────────────────────────────
    feats['corr_xy'] = np.corrcoef(mx, my)[0, 1] if (
        np.std(mx) > 0 and np.std(my) > 0) else 0
    feats['corr_xz'] = np.corrcoef(mx, mz)[0, 1] if (
        np.std(mx) > 0 and np.std(mz) > 0) else 0
    feats['corr_yz'] = np.corrcoef(my, mz)[0, 1] if (
        np.std(my) > 0 and np.std(mz) > 0) else 0

    # ── NEW: Gravity separation (low-pass window=10s) ─────────
    # Critical: pitch/roll_mean separates Class 4 (grav_z=0.64) from others
    GRAV_WIN = 10
    for ax, arr in [('x', mx), ('y', my), ('z', mz)]:
        grav = pd.Series(arr).rolling(
            GRAV_WIN, center=True, min_periods=1).mean().values
        dynamic = arr - grav
        feats[f'grav_{ax}_mean'] = grav.mean()
        feats[f'grav_{ax}_std'] = grav.std()
        feats[f'dynamic_{ax}_mean'] = np.abs(dynamic).mean()
        feats[f'dynamic_{ax}_std'] = dynamic.std()
        feats[f'dynamic_{ax}_energy'] = np.mean(dynamic**2)
        feats[f'dynamic_{ax}_max'] = np.max(np.abs(dynamic))

    # Tilt angles from gravity vector
    gx_lp = pd.Series(mx).rolling(
        GRAV_WIN*2, center=True, min_periods=1).mean().values
    gy_lp = pd.Series(my).rolling(
        GRAV_WIN*2, center=True, min_periods=1).mean().values
    gz_lp = pd.Series(mz).rolling(
        GRAV_WIN*2, center=True, min_periods=1).mean().values
    g_mag = np.sqrt(gx_lp**2 + gy_lp**2 + gz_lp**2) + 1e-9
    pitch = np.arcsin(np.clip(gx_lp / g_mag, -1, 1))
    roll = np.arcsin(np.clip(gy_lp / g_mag, -1, 1))

    feats['pitch_mean'] = pitch.mean()
    feats['pitch_std'] = pitch.std()
    feats['pitch_range'] = np.ptp(pitch)
    feats['pitch_q25'] = np.percentile(pitch, 25)
    feats['pitch_q75'] = np.percentile(pitch, 75)
    feats['roll_mean'] = roll.mean()
    feats['roll_std'] = roll.std()
    feats['roll_range'] = np.ptp(roll)
    feats['roll_q25'] = np.percentile(roll, 25)
    feats['roll_q75'] = np.percentile(roll, 75)

    # ── NEW: FFT on magnitude (proven discriminative from A6) ──
    for col, arr in [('x', mx), ('y', my), ('z', mz), ('mag', mag)]:
        centered = arr - np.mean(arr)
        fft_vals = np.abs(np.fft.rfft(centered))
        freqs = np.fft.rfftfreq(len(arr), d=1.0)
        psd = fft_vals**2
        total_psd = psd.sum() + 1e-9

        feats[f'fft_{col}_max'] = np.max(fft_vals)
        feats[f'fft_{col}_mean'] = np.mean(fft_vals)
        feats[f'fft_{col}_std'] = np.std(fft_vals)
        feats[f'fft_{col}_dominant'] = freqs[np.argmax(
            fft_vals[1:]) + 1]  # dominant freq in Hz
        feats[f'fft_{col}_energy'] = np.sum(psd)

        # Band power ratios (critical: Class 1 dom=0.013 Hz, Class 4 dom=0.28 Hz)
        bands = {
            'vlow':  (freqs >= 0.005) & (freqs < 0.02),
            'low':   (freqs >= 0.02) & (freqs < 0.05),
            'mid':   (freqs >= 0.05) & (freqs < 0.1),
            'high':  (freqs >= 0.1) & (freqs <= 0.5),
        }
        for bname, mask in bands.items():
            feats[f'fft_{col}_{bname}_ratio'] = psd[mask].sum() / total_psd

        # Spectral entropy
        psd_norm = psd / total_psd
        feats[f'fft_{col}_spec_entropy'] = - \
            np.sum(psd_norm * np.log(psd_norm + 1e-9))

    # ── NEW: std_* column aggregate features ──────────────────
    # std_x/y/z = within-second sensor noise; Class 3/4 have high values
    sx = df['std_x'].values
    sy = df['std_y'].values
    sz = df['std_z'].values
    std_mag = np.sqrt(sx**2 + sy**2 + sz**2)
    feats['std_mag_mean'] = std_mag.mean()
    feats['std_mag_std'] = std_mag.std()
    feats['std_mag_max'] = std_mag.max()
    feats['std_mag_range'] = std_mag.max() - std_mag.min()
    feats['std_total'] = sx.mean() + sy.mean() + sz.mean()

    # ── Segment features (6 × 50s windows) ───────────────────
    N_SEGS = 6
    seg_len = len(df) // N_SEGS
    seg_mag_means = []
    for i in range(N_SEGS):
        seg = df.iloc[i*seg_len:(i+1)*seg_len]
        seg_mx = seg['mean_x'].values
        seg_my = seg['mean_y'].values
        seg_mz = seg['mean_z'].values
        seg_mag = np.sqrt(seg_mx**2 + seg_my**2 + seg_mz**2)
        seg_mag_means.append(seg_mag.mean())

        feats[f'seg{i}_mag_mean'] = seg_mag.mean()
        feats[f'seg{i}_mag_std'] = seg_mag.std()
        feats[f'seg{i}_mean_x'] = seg_mx.mean()
        feats[f'seg{i}_mean_y'] = seg_my.mean()
        feats[f'seg{i}_mean_z'] = seg_mz.mean()
        feats[f'seg{i}_std_mean'] = (
            seg['std_x'].mean() + seg['std_y'].mean() + seg['std_z'].mean()) / 3
        feats[f'seg{i}_jerk_mean'] = np.mean(np.abs(np.diff(seg_mag)))

    # Segment-level temporal features
    seg_arr = np.array(seg_mag_means)
    feats['seg_mag_trend'] = np.polyfit(np.arange(N_SEGS), seg_arr, 1)[0]
    feats['seg_mag_var'] = np.var(seg_arr)
    feats['seg_first_last'] = seg_arr[-1] - seg_arr[0]

    # First / mid / last window averages
    for col in ['mean_x', 'mean_y', 'mean_z']:
        s = df[col].values
        feats[f'{col}_first30'] = np.mean(s[:30])
        feats[f'{col}_mid30'] = np.mean(s[135:165])
        feats[f'{col}_last30'] = np.mean(s[-30:])

    return feats

In [ ]:
def load_dataset(folder, is_train=True):
    records = []
    for fpath in sorted(folder.rglob('*.csv')):
        df = pd.read_csv(fpath)
        fid = int(df['file_id'].iloc[0])
        user = fpath.parent.name          # e.g. "User_001"
        label = int(df['label'].iloc[0]) if is_train else -1
        feats = extract_features(df)
        feats['file_id'] = fid
        feats['label'] = label
        feats['user'] = user
        records.append(feats)
    return pd.DataFrame(records)


print("Loading train...")
train_df = load_dataset(TRAIN_DIR, is_train=True)
print(f"  Train shape: {train_df.shape}")

print("Loading test...")
test_df = load_dataset(TEST_DIR, is_train=False)
print(f"  Test shape : {test_df.shape}")

# Separate out metadata
train_ids = train_df['file_id'].values
train_users = train_df['user'].values
y_train = train_df['label'].values
test_ids = test_df['file_id'].values

feat_cols = [c for c in train_df.columns if c not in [
    'file_id', 'label', 'user']]
X_train = train_df[feat_cols]
X_test = test_df[feat_cols]

print(f"\nFeature count : {len(feat_cols)}")
print(f"Label distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

Loading train...
  Train shape: (11020, 273)
Loading test...
  Test shape : (6849, 273)

Feature count : 270
Label distribution:
0    4643
1    4695
2     358
3     656
4     142
5     526
Name: count, dtype: int64


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler

N_SPLITS = 5

# GroupKFold by user (the correct approach given 0 user overlap)
gkf = GroupKFold(n_splits=N_SPLITS)
cv_splits = list(gkf.split(X_train, y_train, groups=train_users))

# Verify no user leaks
print("=== GroupKFold Verification ===")
for fold_i, (tr_idx, val_idx) in enumerate(cv_splits):
    tr_users = set(train_users[tr_idx])
    val_users = set(train_users[val_idx])
    leak = tr_users & val_users
    val_labels = pd.Series(
        y_train[val_idx]).value_counts().sort_index().to_dict()
    print(f"  Fold {fold_i+1}: train={len(tr_idx)} val={len(val_idx)} "
          f"user_leak={len(leak)} val_label_dist={val_labels}")

# Scaler (fit on train only, applied per fold for GBMs; globally for CNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Class weights for imbalanced learning
class_weights = compute_class_weight(
    'balanced', classes=np.arange(6), y=y_train)
weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f"\nClass weights (balanced): {[f'{w:.2f}' for w in class_weights]}")
print("→ Classes 2,4,5 will be upweighted to compensate for 33x imbalance")

# Sample weights vector (for GBMs)
sample_weights = np.array([weight_dict[y] for y in y_train])

=== GroupKFold Verification ===
  Fold 1: train=8831 val=2189 user_leak=0 val_label_dist={0: 947, 1: 985, 2: 82, 3: 100, 4: 21, 5: 54}
  Fold 2: train=8813 val=2207 user_leak=0 val_label_dist={0: 892, 1: 918, 2: 94, 3: 135, 4: 16, 5: 152}
  Fold 3: train=8807 val=2213 user_leak=0 val_label_dist={0: 864, 1: 1007, 2: 58, 3: 138, 4: 16, 5: 130}
  Fold 4: train=8823 val=2197 user_leak=0 val_label_dist={0: 922, 1: 906, 2: 57, 3: 141, 4: 46, 5: 125}
  Fold 5: train=8806 val=2214 user_leak=0 val_label_dist={0: 1018, 1: 879, 2: 67, 3: 142, 4: 43, 5: 65}

Class weights (balanced): ['0.40', '0.39', '5.13', '2.80', '12.93', '3.49']
→ Classes 2,4,5 will be upweighted to compensate for 33x imbalance


In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score

oof_lgb = np.zeros((len(X_train), 6))
test_lgb = np.zeros((len(X_test),  6))

lgb_params = dict(
    objective='multiclass',
    num_class=6,
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    class_weight=weight_dict,   # KEY: handles 33x imbalance
)

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train[tr_idx], y_train[val_idx]
    wtr = sample_weights[tr_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        Xtr, ytr,
        sample_weight=wtr,
        eval_set=[(Xval, yval)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )
    oof_lgb[val_idx] = model.predict_proba(Xval)
    test_lgb += model.predict_proba(X_test) / N_SPLITS

    fold_f1 = f1_score(yval, oof_lgb[val_idx].argmax(1), average='macro')
    print(
        f"  Fold {fold+1} LGB F1: {fold_f1:.4f}  best_iter={model.best_iteration_}")

lgb_oof_f1 = f1_score(y_train, oof_lgb.argmax(1), average='macro')
print(f"\nLGB OOF F1: {lgb_oof_f1:.4f}")
print(
    f"Per-class : {f1_score(y_train, oof_lgb.argmax(1), average=None).round(4)}")

  Fold 1 LGB F1: 0.6479  best_iter=168
  Fold 2 LGB F1: 0.7221  best_iter=194
  Fold 3 LGB F1: 0.7339  best_iter=170
  Fold 4 LGB F1: 0.7067  best_iter=135
  Fold 5 LGB F1: 0.6618  best_iter=168

LGB OOF F1: 0.7069
Per-class : [0.9616 0.8819 0.2623 0.674  0.8321 0.6295]


In [ ]:
import xgboost as xgb
from sklearn.metrics import f1_score

oof_xgb = np.zeros((len(X_train), 6))
test_xgb = np.zeros((len(X_test),  6))

xgb_params = dict(
    objective='multi:softprob',
    num_class=6,
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    device='cuda',   # remove if no GPU
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    verbosity=0,
)

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train[tr_idx], y_train[val_idx]
    wtr = sample_weights[tr_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        Xtr, ytr,
        sample_weight=wtr,
        eval_set=[(Xval, yval)],
        verbose=False,
    )
    oof_xgb[val_idx] = model.predict_proba(Xval)
    test_xgb += model.predict_proba(X_test) / N_SPLITS

    fold_f1 = f1_score(yval, oof_xgb[val_idx].argmax(1), average='macro')
    print(
        f"  Fold {fold+1} XGB F1: {fold_f1:.4f}  best_iter={model.best_iteration}")

xgb_oof_f1 = f1_score(y_train, oof_xgb.argmax(1), average='macro')
print(f"\nXGB OOF F1: {xgb_oof_f1:.4f}")
print(
    f"Per-class : {f1_score(y_train, oof_xgb.argmax(1), average=None).round(4)}")

  Fold 1 XGB F1: 0.6658  best_iter=361
  Fold 2 XGB F1: 0.7480  best_iter=387
  Fold 3 XGB F1: 0.7297  best_iter=335
  Fold 4 XGB F1: 0.7037  best_iter=269
  Fold 5 XGB F1: 0.6637  best_iter=374

XGB OOF F1: 0.7137
Per-class : [0.9639 0.8963 0.2594 0.6814 0.8222 0.6591]


In [ ]:
CHANNELS = ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']
SEQ_LEN = 300


def build_sequences(folder, is_train=True):
    seqs, lbls, users = [], [], []
    for fpath in sorted(folder.rglob('*.csv')):
        df = pd.read_csv(fpath)
        user = fpath.parent.name
        seq = df[CHANNELS].values.astype(np.float32)   # (300, 6)
        seqs.append(seq)
        lbls.append(int(df['label'].iloc[0]) if is_train else -1)
        users.append(user)
    return np.array(seqs), np.array(lbls), np.array(users)


print("Building sequences...")
train_seq, train_lbl, train_seq_users = build_sequences(
    TRAIN_DIR, is_train=True)
test_seq,  _,         _ = build_sequences(TEST_DIR,  is_train=False)

# Normalize per channel using train statistics
seq_mean = train_seq.mean(axis=(0, 1), keepdims=True)  # (1, 1, 6)
seq_std = train_seq.std(axis=(0, 1),  keepdims=True) + 1e-9
train_seq = (train_seq - seq_mean) / seq_std
test_seq = (test_seq - seq_mean) / seq_std

# Transpose to (N, channels, seq_len) for Conv1d
train_seq = train_seq.transpose(0, 2, 1)  # (N, 6, 300)
test_seq = test_seq.transpose(0, 2, 1)

print(f"train_seq: {train_seq.shape}")
print(f"test_seq : {test_seq.shape}")
print(
    f"train_lbl distribution: {pd.Series(train_lbl).value_counts().sort_index().to_dict()}")

Building sequences...
train_seq: (11020, 6, 300)
test_seq : (6849, 6, 300)
train_lbl distribution: {0: 4643, 1: 4695, 2: 358, 3: 656, 4: 142, 5: 526}


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

# ── Focal Loss ────────────────────────────────────────────────


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        n_cls = logits.size(1)
        # Label smoothing
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits).scatter_(
                1, targets.unsqueeze(1), 1.0)
            smooth_targets = smooth_targets * (1 - self.label_smoothing) \
                + self.label_smoothing / n_cls

        log_prob = torch.log_softmax(logits, dim=1)
        prob = torch.exp(log_prob)
        # Focal weight
        pt = (smooth_targets * prob).sum(dim=1)
        focal_w = (1 - pt) ** self.gamma
        loss = -(smooth_targets * log_prob).sum(dim=1)
        if self.weight is not None:
            cls_w = self.weight[targets]
            loss = loss * cls_w
        return (focal_w * loss).mean()

# ── Residual Block ────────────────────────────────────────────


class ResBlock(nn.Module):
    def __init__(self, channels, kernel=3):
        super().__init__()
        pad = kernel // 2
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel, padding=pad),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Conv1d(channels, channels, kernel, padding=pad),
            nn.BatchNorm1d(channels),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))

# ── Model ─────────────────────────────────────────────────────


class HARNet(nn.Module):
    def __init__(self, in_ch=6, n_classes=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),   # 150
        )
        self.stage1 = nn.Sequential(
            ResBlock(64), ResBlock(64),
            nn.Conv1d(64, 128, 3, padding=1, stride=2),  # 75
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.stage2 = nn.Sequential(
            ResBlock(128), ResBlock(128),
            nn.Conv1d(128, 256, 3, padding=1, stride=2),  # 38
            nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.stage3 = nn.Sequential(
            ResBlock(256), ResBlock(256),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return self.head(x)


# ── Training Loop ─────────────────────────────────────────────
oof_cnn = np.zeros((len(train_seq), 6))
test_cnn = np.zeros((len(test_seq),  6))

EPOCHS = 150
BATCH = 128

cls_weights_tensor = torch.tensor(
    class_weights, dtype=torch.float32).to(DEVICE)
criterion = FocalLoss(
    gamma=2.0, weight=cls_weights_tensor, label_smoothing=0.05)

Xt = torch.tensor(test_seq).to(DEVICE)

# Use same GroupKFold splits (aligned with GBM splits by file order)
# Rebuild splits on sequence arrays (same order as load_dataset)
seq_splits = list(GroupKFold(n_splits=N_SPLITS).split(
    train_seq, train_lbl, groups=train_seq_users))

for fold, (tr_idx, val_idx) in enumerate(seq_splits):
    print(f'\n--- Fold {fold+1}/{N_SPLITS} ---')

    Xtr = torch.tensor(train_seq[tr_idx]).to(DEVICE)
    ytr = torch.tensor(train_lbl[tr_idx]).long().to(DEVICE)
    Xval = torch.tensor(train_seq[val_idx]).to(DEVICE)
    yval_np = train_lbl[val_idx]

    ds = TensorDataset(Xtr, ytr)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)

    model = HARNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_f1, best_probs, best_state = 0, None, None

    for ep in range(EPOCHS):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            nn.utils.clip_grad_norm_(
                model.parameters(), 1.0)  # gradient clipping
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_probs = torch.softmax(model(Xval), dim=1).cpu().numpy()
        ep_f1 = f1_score(yval_np, val_probs.argmax(1), average='macro')

        if ep_f1 > best_f1:
            best_f1 = ep_f1
            best_probs = val_probs.copy()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (ep + 1) % 25 == 0:
            print(
                f'  Epoch {ep+1}/{EPOCHS} | F1: {ep_f1:.4f} | Best: {best_f1:.4f}')

    oof_cnn[val_idx] = best_probs
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_cnn += torch.softmax(model(Xt), dim=1).cpu().numpy() / N_SPLITS

    print(f'Fold {fold+1} CNN Best F1: {best_f1:.4f}')

cnn_oof_f1 = f1_score(train_lbl, oof_cnn.argmax(1), average='macro')
print(f'\nCNN OOF F1: {cnn_oof_f1:.4f}')
print(
    f'Per-class : {f1_score(train_lbl, oof_cnn.argmax(1), average=None).round(4)}')

Device: cuda

--- Fold 1/5 ---
  Epoch 25/150 | F1: 0.6242 | Best: 0.6419
  Epoch 50/150 | F1: 0.6314 | Best: 0.6692
  Epoch 75/150 | F1: 0.6487 | Best: 0.6692
  Epoch 100/150 | F1: 0.6550 | Best: 0.6707
  Epoch 125/150 | F1: 0.6768 | Best: 0.6768
  Epoch 150/150 | F1: 0.6641 | Best: 0.6768
Fold 1 CNN Best F1: 0.6768

--- Fold 2/5 ---
  Epoch 25/150 | F1: 0.6084 | Best: 0.7154
  Epoch 50/150 | F1: 0.6487 | Best: 0.7154
  Epoch 75/150 | F1: 0.7120 | Best: 0.7289
  Epoch 100/150 | F1: 0.7075 | Best: 0.7289
  Epoch 125/150 | F1: 0.7179 | Best: 0.7305
  Epoch 150/150 | F1: 0.7078 | Best: 0.7305
Fold 2 CNN Best F1: 0.7305

--- Fold 3/5 ---
  Epoch 25/150 | F1: 0.6601 | Best: 0.7079
  Epoch 50/150 | F1: 0.7068 | Best: 0.7306
  Epoch 75/150 | F1: 0.7206 | Best: 0.7306
  Epoch 100/150 | F1: 0.7031 | Best: 0.7306
  Epoch 125/150 | F1: 0.6700 | Best: 0.7306
  Epoch 150/150 | F1: 0.6759 | Best: 0.7306
Fold 3 CNN Best F1: 0.7306

--- Fold 4/5 ---
  Epoch 25/150 | F1: 0.6404 | Best: 0.6830
  Epoch 

In [ ]:
from scipy.optimize import minimize


def ensemble_f1_neg(weights):
    w = np.abs(weights) / (np.abs(weights).sum() + 1e-9)
    blended = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cnn
    return -f1_score(y_train, blended.argmax(1), average='macro')


best_score, best_weights = 1.0, [0.33, 0.33, 0.34]
for _ in range(50):
    w0 = np.random.dirichlet(np.ones(3))
    res = minimize(ensemble_f1_neg, w0, method='Nelder-Mead',
                   options={'maxiter': 3000, 'xatol': 1e-6})
    if res.fun < best_score:
        best_score, best_weights = res.fun, res.x

w = np.abs(best_weights) / np.abs(best_weights).sum()
print(f'Optimal weights: LGB={w[0]:.3f}  XGB={w[1]:.3f}  CNN={w[2]:.3f}')

oof_ensemble = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cnn
ensemble_f1 = f1_score(y_train, oof_ensemble.argmax(1), average='macro')
print(f'Ensemble OOF F1: {ensemble_f1:.4f}')
print(
    f'Per-class: {f1_score(y_train, oof_ensemble.argmax(1), average=None).round(4)}')

print('\n=== Individual Model OOF F1 ===')
for name, oof in [('LGB', oof_lgb), ('XGB', oof_xgb), ('CNN', oof_cnn)]:
    mf1 = f1_score(y_train, oof.argmax(1), average='macro')
    pcf1 = f1_score(y_train, oof.argmax(1), average=None).round(3)
    print(f'  {name}: macro={mf1:.4f}  per-class={pcf1}')

Optimal weights: LGB=0.061  XGB=0.478  CNN=0.462
Ensemble OOF F1: 0.7332
Per-class: [0.9641 0.9036 0.2871 0.7145 0.8162 0.7135]

=== Individual Model OOF F1 ===
  LGB: macro=0.7069  per-class=[0.962 0.882 0.262 0.674 0.832 0.629]
  XGB: macro=0.7137  per-class=[0.964 0.896 0.259 0.681 0.822 0.659]
  CNN: macro=0.7071  per-class=[0.957 0.886 0.279 0.68  0.747 0.694]


In [ ]:
test_ensemble = w[0]*test_lgb + w[1]*test_xgb + w[2]*test_cnn
test_preds = test_ensemble.argmax(axis=1)

sample_sub = pd.read_csv(SAMPLE)
pred_map = dict(zip(test_ids.tolist(), test_preds.tolist()))
sample_sub['Label'] = sample_sub['Id'].map(pred_map)

assert sample_sub['Label'].isna().sum() == 0, "Some IDs were not predicted!"

sample_sub.to_csv('submission.csv', index=False)
print('submission.csv written')
print(sample_sub['Label'].value_counts().sort_index())

submission.csv written
Label
0    2800
1    3133
2      82
3     511
4      79
5     244
Name: count, dtype: int64
